# Interactive Tutorial for the HWO UV Instrument (UVI) 

[SYOTools](https://github.com/spacetelescope/hwo-tools) is a framework to enable users to create [Science Yield Optimization web tools](http://hwo.stsci.edu) for observatory design. It uses Bokeh & astropy to visualize exposure time calculators (ETCs), as well as other science cases, as a function of various observatory parameters. While SYOTools can be used independently of a particular observatory or science case, it has been primarily written to facilitate the design of [HWO](https://www.habitableworldsobservatory.org).

SYOTools is divided into two main parts, contained in the `syotools.models` and `syotools.interface` subpackages. This Jupyter Notebook is intended as a walkthrough for using `syotools.models` to perform calculations. A future tutorial on using `syotools.interface` to design a web tool is in preparation; however, it will not be a similar Jupyter Notebook, as the interface framework is highly integrated with Bokeh Server. Instead, we will use Bokeh's `output_notebook` function, along with IPython interactors (as described [here](https://github.com/bokeh/bokeh/blob/0.12.13/examples/howto/notebook_comms/Jupyter%20Interactors.ipynb)), to visualize the example calculations below. 

In [1]:
#Import Bokeh interface tools
from ipywidgets import interact
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure
from bokeh.layouts import column
output_notebook()

#Import numpy, syotools, and astropy.units
### NOTE: you will likely receive some pysynphot warnings when executing these imports, these are expected
import numpy as np
from syotools.models import Telescope, Source, Spectrograph, SourceSpectrographicExposure #models for the observatory and instruments
from syotools.utils.jsonunit import str_jsunit #for printing JsonUnit and JsonSpectrum wrappers in a readable way
from syotools.utils import pre_encode, pre_decode
from syotools.spectra import SpectralLibrary
import astropy.units as u
import synphot as syn

Loading BokehJS ...

## Example 2: UVI ETC

*"The UV Instrument (HWO UVI) is a multi-object spectrograph covering far-ultraviolet (100 nm) through visible (850 nm) wavelengths. UVI is a highly multiplexed ultraviolet spectrograph, with medium and low-resolution multi-object imaging spectroscopy and FUV imaging modes. UVI can be considered as a successor to the Hubble Space Telescope Imaging Spectrograph (STIS) instrument, with two orders-of-magnitude higher efficiency, multi-object capability, and a wide-field multi-band imaging channel."* Adapted from the LUVOIR STDT study of LUMOS

In this example, we will create a exposure time calculator for UVI, so that we can calculate the signal-to-noise ratio (SNR) for several possible template spectra. This approximates some of the functionality of [the official UVI ETC tool](http://hwo.stsci.edu/uvipec_etc).

In [2]:
#Again, instatiate the observatory using syotools.models.Telescope
hwo_ex1 = Telescope()
hwo_ex1.set_from_hwome('EAC5') 

In [3]:
# Select the instrument, already linked to the telescope.
uvi = hwo_ex1.instruments["UV_MOS.NUV_MOS_MOS"]

In [4]:
#Create a new spectrographic exposure for the camera
uvi_exposure = uvi.create_exposure()

In [5]:
# Customize the source and exposure

# Create the source
uvi_sed = Source()
# Configure the source
template = "QSO"
fuvmag = 21
bandpass = "galex,fuv"
redshift = 0
extinction = 0
uvi_sed.set_sed(template, fuvmag, redshift, extinction, bandpass=bandpass)


# Add the source to the SourceExposure
uvi_exposure.source = uvi_sed

#Print the available grating modes
available_uvi_gratings = [band for band in uvi.bandnames]
print("UVI grating modes:") 
for grating in available_uvi_gratings:
    print("   "+grating) 

#Print the default template & mode
default_uvi_template = SpectralLibrary[uvi_exposure.source.name]
default_uvi_band = uvi.band
print("Current SED template: {}".format(default_uvi_template)) 
print("Current grating mode: {}".format(default_uvi_band)) 

uvi_template_codes = ['Classical T Tauri', 'M1 Dwarf', 'G Dwarf', '10 Myr Starburst', 
            'QSO', 'Seyfert 1', 'Seyfert 2', 'Liner', 'O5V Star', 'G2V Star', 'Orion Nebula', 'G191B2B (WD)',
    	    'GD71 (WD)', 'GD153 (WD)', 'Starburst, No Dust', 'Starburst, E(B-V) = 0.6', 'B5V Star', 
            'M2V Star', 'Elliptical Galaxy', 'Sbc Galaxy', 'Starburst Galaxy', 'NGC 1068', 
            'Galaxy with f_esc, HI=1, HeI=1', 'Galaxy with f_esc, HI=0.001, HeI=1', 'Blackbody5000', 'Blackbody100000' ]
available_uvi_templates = [SpectralLibrary[tc] for tc in uvi_template_codes]

UVI grating modes:
   NUV_MOS.G300M
   NUV_MOS.G650LL
   NUV_MOS.G700L
Current SED template: QSO
Current grating mode: None


In [6]:
#Create the Bokeh SED figure
uvi_bgflux = syn.units.convert_flux(uvi.sky.waveset, uvi.sky(uvi.sky.waveset), u.erg / u.s / u.cm**2 / u.AA)
uvi_bgwave = uvi.sky.waveset.to_value(u.AA)
uvi_sed_fig = figure(height=300, width=600, title="SED", x_axis_label="Wavelength [Angstrom]",
                 y_axis_label="Flux [erg / s / cm2 / Ang]", y_range=(0., 4.0e-16), x_range=(900, 4000))
uvi_sed_flux = syn.units.convert_flux(uvi_sed.sed.waveset, uvi_sed.sed(uvi_sed.sed.waveset), syn.units.FLAM)
uvi_sed_line = uvi_sed_fig.line(uvi_sed.sed.waveset.to_value(u.AA), uvi_sed_flux.value, color='firebrick', 
                                    line_width=3) #legend='Source Flux')
uvi_bef_line = uvi_sed_fig.line(uvi_bgwave, uvi_bgflux, color='darksalmon', 
                                    line_width=3) #  legend='Background')

In [7]:
#Create the Bokeh SNR figure
uvi_snr = uvi_exposure.recover('snr')
uvi_wave = uvi_exposure.recover('wave')
uvi_snr_fig = figure(height=300, width=600, title="SNR", x_axis_label="Wavelength [Angstrom]", 
                 y_axis_label="S/N per resel", y_range=(0.0, 40.0), x_range=(900, 4000))
uvi_snr_line = uvi_snr_fig.line(uvi_wave[0], uvi_snr[0], color='orange', line_width=3)

In [8]:
#Define the update callback function for interactive inputs
def uvi_update(template=default_uvi_template, aperture=15., exptime=1., 
                 v_magnitude=21., band=default_uvi_band):

    #turn off calculations until everything is updated
    uvi_exposure.disable()
    
    #find the correct template code & grating mode
    uvi_exposure.source.set_sed(template, fuvmag, redshift, extinction, bandpass=bandpass)
    
    #update all of the telescope, spectrograph, & exposure parameters
    hwo_ex1.effective_diameter = aperture * u.m
    uvi.band = band
    uvi_exposure.exptime = exptime * u.h
    
    #turn calculations back on and recalculate based on updated parameters
    uvi_exposure.unknown = "snr"
    
    #recover the recalculated values, and make sure everything is in the right units
    uvi_sed, uvi_snr, uvi_wave = uvi_exposure.recover('source', 'snr', 'wave')
    wave = uvi_sed.sed.waveset.to(u.AA)
    sky = uvi.recover('sky')
    uvi_bgflux = syn.units.convert_flux(wave, sky(wave), (u.erg / u.s / u.cm**2 / u.AA)).value
    uvi_bgwave = wave.to_value(u.AA)
    
    #sanitize the fluxes and SNR because some of the spectra don't play nice
    uvi_flux = syn.units.convert_flux(wave, uvi_sed.sed(wave), syn.units.FLAM).value
    uvi_flux[~np.isfinite(uvi_flux)] = 0.
    uvi_bgflux[~np.isfinite(uvi_bgflux)] = 0.
    uvi_snr[0][~np.isfinite(uvi_snr[0])] = 0.
    uvi_wave[0][~np.isfinite(uvi_snr[0])] = 0 # intentional, trim the wavelength array
    
    #update the SED figure
    uvi_sed_fig.y_range.start = 0.
    uvi_sed_fig.y_range.end = 1.5 * uvi_flux.max()
    uvi_sed_line.data_source.data = {'x': wave.value, 'y': uvi_flux}
    uvi_bef_line.data_source.data = {'x': wave.value, 'y': uvi_bgflux}
    
    #update the SNR figure
    uvi_snr_fig.y_range.start = 0.
    uvi_snr_fig.y_range.end = 1.5 * uvi_snr[0].value.max()
    uvi_snr_line.data_source.data = {'x': uvi_wave[0].value, 'y': uvi_snr[0].value}
    
    #update the plots
    push_notebook()

In [22]:
#Show the plots
# depending on your jupyter config you may need to shift+enter this cell 
# after changing the sliders in the cell below to see an updated result plot
uvi_handle = show(column(uvi_sed_fig, uvi_snr_fig), notebook_handle=True)

In [21]:
#Create the interactive inputs
#shift+enter here will reset to defaults. 
uvi_inputs = interact(uvi_update, template=available_uvi_templates, aperture=(2.0, 12.0), 
                        exptime=(0.1, 10.0, 0.1), v_magnitude=(15.0, 30.0, 0.1), band=available_uvi_gratings)

interactive(children=(Dropdown(description='template', index=4, options=('Classical T-Tauri Star', 'M1 Dwarf',…